# LLM & SerpAPI

In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_community.utilities import SerpAPIWrapper

load_dotenv()

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
search = SerpAPIWrapper()

def test_serpapi():
    query = "أحدث أخبار نماذج الذكاء الاصطناعي 2026"
    print(f"🔍 بنجرب نبحث عن: {query}\n")
    
    result = search.run(query)
    
    print("📝 نتيجة البحث من SerpAPI:")
    print(result)

test_serpapi()

print("LLM & SerpAPI✅")

🔍 بنجرب نبحث عن: أحدث أخبار نماذج الذكاء الاصطناعي 2026

📝 نتيجة البحث من SerpAPI:
['أحدث الذكاء الاصطناعي في 2026 انعطافة تاريخية في المجال الصحي، إذ انتقلنا من نموذج الطب القائم على "المتوسطات الإحصائية العامة" إلى "الطب ...', 'تحدثت المراسلة Anabelle Nicoud مع عدة خبراء في مجالات الذكاء الاصطناعي، والأمن، والحوسبة الكمية، وغير ذلك، لفهم أفضل لتوجُّهات التكنولوجيا في ...', 'في هذا الفيديو المنتظر، نضع النقاط على الحروف ونكشف عن الترتيب النهائي والحقيقي لأقوى نماذج الذكاء الاصطناعي (AI Models) مع دخولنا عام 2026.', '* إذا كنت تبحث عن الأمان الوظيفي: اتجه فوراً لـ الأمن السيبراني. * إذا كنت شغوفاً بالتكنولوجيا والابتكار: ابدأ بـ دورات الذكاء الاصطناعي الـ 10. * ...', 'استكشف آخر أخبار الذكاء الاصطناعي 2026 – من الوكلاء الرقميين المستقلين إلى ثورة الطب والرقائق العصبية · جيل الوكلاء الرقميين: ما وراء الشات بوت · مقارنة بين أقوى ...', '7 توقعات كبرى للذكاء الاصطناعي في 2026 · 1. Anthropic إلى البورصة وOpenAI تؤجل الاكتتاب العام · 2. تسريب أبحاث مختبر SSI يهز المشهد العالمي · 3. تقدم تدري

# Agents

In [11]:
from typing import TypedDict, List, Annotated
import operator
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

#1- هنا بنعمل ملف مشترك علشان الايجنتس هيباصو لبعض فيه
class ResearchState(TypedDict): 
    query: str # سؤال المستخدم
    plan: List[str] # الاسئلة الي بلانر بيحطها
    research_data: Annotated[List[str], operator.add] # الداتا الي الريشيرشر بيجمعها عن الاسئلة
    draft: str # المكان الي الكاتب بيكتب فيه
    feedback: str # زي ميتا داتا او تعليقات المراجع
    revision_count: int # عدد مرات التعديل علشان منخشش في حلقة مفرغة
    
#2- First Agent (Planner)
def planner_node(state: ResearchState):
    print("👨‍💼 Planner: Creating a research plan...")
    prompt = f"""You are a research planner. Your task is to break down the following topic into 3 precise and concise research questions suitable for Google search.
    Topic: {state['query']}
    Return only the questions, each on a separate line, without any numbering or introductions."""
    
    response= llm.invoke([HumanMessage(content=prompt)])
    plan = [q.strip() for q in response.content.split('\n') if q.strip()]
    return {"plan": plan, "revision_count": state.get("revision_count", 0)}

#3- Sec Agent (Researcher)
def researcher_node(state: ResearchState):
    print(f"🕵️‍♂️ Researcher: Searching across {len(state['plan'])} topics...")
    gathered_data = []

    for q in state['plan']:
        print(f"   - Searching for: {q}")
        try:
            result = search.run(q)
            gathered_data.append(f"Search results for '{q}':\n{result}")
        except Exception as e:
            gathered_data.append(f"Error while searching for '{q}'")
                     
    return {"research_data": gathered_data}

#4- Third Agent (Writer)
def writer_node(state: ResearchState):
    print("✍️ Writer: Generating the report...")
    data_str = "\n\n".join(state['research_data'])

    prompt = f"""You are a professional report writer. Based on the following information, write a comprehensive and well-structured report on the requested topic.
    Requested topic: {state['query']}
    Available information: {data_str}

    Reviewer notes (if any): {state.get('feedback', 'No previous feedback')}
    """
    
    response= llm.invoke([HumanMessage(content=prompt)])
    return {"draft": response.content}

#5- Fourth Agent (Critic)
def critic_node(state: ResearchState):
    print("🧐 Critic: Reviewing the report...")

    prompt = f"""You are a strict quality reviewer. Read the following report and ensure it fully and accurately answers the original question.
    Original question: {state['query']}
    Report: {state['draft']}

    If the report is excellent and complete, write only the word "Approved" at the beginning of your response.
    If it needs revision, clearly provide feedback for the writer to improve it."""
    
    response = llm.invoke([HumanMessage(content=prompt)])
    feedback = response.content.strip()
    
    new_count = state.get("revision_count", 0) + 1
    
    # نستخدم in عشان لو الموديل حط نقطة أو مسافة جنب الكلمة
    if "Approved" in feedback or "approved" in feedback.lower() or "مقبول" in feedback or new_count >= 2:
        return {"feedback": "Approved", "revision_count": new_count}
    else:
        print("   ❌ The critic rejected the report and requested revisions!")
        return {"feedback": feedback, "revision_count": new_count}

print("✅ The team is ready (Planner, Researcher, Writer, Critic)!")

✅ The team is ready (Planner, Researcher, Writer, Critic)!


# Agents Graph

In [ ]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(ResearchState)

# بنضيف ال Agents للجراف
workflow.add_node("planner", planner_node)
workflow.add_node("researcher", researcher_node)
workflow.add_node("writer", writer_node)
workflow.add_node("critic", critic_node)

# بنظبط الطريق لل Agents 
workflow.add_edge(START, "planner")
workflow.add_edge("planner", "researcher")
workflow.add_edge("researcher", "writer")
workflow.add_edge("writer", "critic")

# بنعمل شرط للمراجع: لو التقرير مقبول يقفل. لو مرفوض نرجع للكاتب تاني
def critic_router(state: ResearchState):
    if state.get("revision_count", 0) >= 2 or "مقبول" in state["feedback"] or "Approved" in state["feedback"]:
        return END
    else:
        return "writer"  # يرجع للكاتب يعدل

workflow.add_conditional_edges(
    "critic",
    critic_router,
 # بنعرفه يروح فين
    )

# Compile بنعمل
research_agent = workflow.compile()
print("✅ The Multi-Agent Graph has been successfully built!")

✅ The Multi-Agent Graph has been successfully built!


# Test

In [13]:
query = "What are the latest developments in generative AI and Agentic AI in 2026?"

print(f"🚀 Starting task: {query}\n" + "="*50)

final_state = research_agent.invoke({
    "query": query,
    "research_data": [],
    "revision_count": 0
})

print("="*50 + "\nالتقرير النهائي:\n")
print(final_state.get("draft", "لم يتم إنشاء مسودة."))

🚀 Starting task: What are the latest developments in generative AI and Agentic AI in 2026?
👨‍💼 Planner: Creating a research plan...
🕵️‍♂️ Researcher: Searching across 3 topics...
   - Searching for: What are the key advancements in generative AI models released in 2026?
   - Searching for: What are the current applications and use cases of Agentic AI in industries such as healthcare and finance in 2026?
   - Searching for: How do recent breakthroughs in Agentic AI and generative AI intersect and influence each other in 2026?
✍️ Writer: Generating the report...
🧐 Critic: Reviewing the report...
   ❌ The critic rejected the report and requested revisions!
✍️ Writer: Generating the report...
🧐 Critic: Reviewing the report...
التقرير النهائي:

**Comprehensive Report: Latest Developments in Generative AI and Agentic AI in 2026**

**Executive Summary**

Generative AI and agentic AI have made significant advancements in 2026, transforming industries such as healthcare, finance, and education.